In [1]:
# ── Cell 1: Install ──────────────────────────────────────────────────────────
!pip install -q catboost scikit-learn pandas numpy


[notice] A new release of pip is available: 26.0 -> 26.0.1
[notice] To update, run: python3.14 -m pip install --upgrade pip


In [2]:
# ── Cell 2: Imports ──────────────────────────────────────────────────────────
import pandas as pd
import numpy as np
import warnings
warnings.filterwarnings('ignore')

from catboost import CatBoostClassifier
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import balanced_accuracy_score, classification_report
from scipy.optimize import minimize

SEEDS    = [42, 7, 123]
N_SPLITS = 10
print('Libraries loaded.')

Libraries loaded.


In [3]:
# ── Cell 3: Load data ────────────────────────────────────────────────────────
TRAIN_DATA  = pd.read_csv('train-data.csv',  index_col='id')
TRAIN_LABEL = pd.read_csv('train-label.csv', index_col='id')
TEST_DATA   = pd.read_csv('test-data.csv',   index_col='id')

print(f'Train: {TRAIN_DATA.shape}, Test: {TEST_DATA.shape}')
print(TRAIN_LABEL['disorder'].value_counts().sort_index())

Train: (13249, 41), Test: (8834, 41)
disorder
0     389
1    2068
2    1090
3    3096
4      58
5    1700
6     813
7    2643
8      91
9    1301
Name: count, dtype: int64


In [4]:
# ── Cell 4: Preprocessing (a3 original) ─────────────────────────────────────
def preprocess(df):
    df = df.copy()

    drop_cols = [
        'first_name', 'last_name', 'insitute_name', 'institute_location',
        'test_1', 'test_2', 'test_3', 'test_4', 'test_5', 'treatment_consent'
    ]
    df = df.drop(columns=drop_cols)

    # Missing flags before encoding
    miss_cols = [
        'gender', 'maternal_defect', 'mother_age', 'father_age',
        'respiration', 'heart_rate', 'risk_level', 'place_birth',
        'folic_acid', 'maternal_illness', 'infertility_treatment',
        'problem_previous_pregnancies', 'abortion_cnt',
        'birth_defects', 'white_blood_cell_count', 'blood_test',
        'symptom_1', 'symptom_2', 'symptom_3', 'symptom_4', 'symptom_5'
    ]
    df['missing_count']      = df[miss_cols].isna().sum(axis=1)
    df['missing_parent_age'] = df['mother_age'].isna().astype(int) + df['father_age'].isna().astype(int)
    df['missing_symptoms']   = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].isna().sum(axis=1)
    df['missing_clinical']   = df[['respiration','heart_rate','risk_level','blood_test']].isna().sum(axis=1)

    for col in ['mother_age','father_age','maternal_defect','gender',
                'risk_level','heart_rate','respiration','abortion_cnt','white_blood_cell_count']:
        df[f'{col}_missing'] = df[col].isna().astype(int)

    # Encode
    binary_yn = [
        'mother_defect','father_defect','maternal_defect','paternal_defect',
        'alive','folic_acid','maternal_illness','infertility_treatment',
        'problem_previous_pregnancies',
        'symptom_1','symptom_2','symptom_3','symptom_4','symptom_5'
    ]
    for col in binary_yn:
        df[col] = df[col].map({'Y': 1, 'N': 0})

    df['respiration']   = df['respiration'].map({'A': 1, 'N': 0})
    df['heart_rate']    = df['heart_rate'].map({'A': 1, 'N': 0})
    df['risk_level']    = df['risk_level'].map({'H': 1, 'L': 0})
    df['place_birth']   = df['place_birth'].map({'I': 1, 'H': 0})
    df['birth_defects'] = df['birth_defects'].map({'S': 1, 'M': 2})
    df['gender']        = df['gender'].map({'M': 0, 'F': 1, 'A': 2})
    df['autopsy']       = df['autopsy'].map({'Y': 1, 'N': 0})

    for col in ['birth_asphyxia', 'radiation_exposure', 'substance_abuse']:
        df[col] = df[col].map({'Y': 1, 'N': 0, 'NR': 2})

    df['blood_test'] = df['blood_test'].map({'N': 0, 'I': 1, 'S': 2, 'A': 3})

    # Engineered features
    df['defect_sum']  = df[['mother_defect','father_defect','maternal_defect','paternal_defect']].sum(axis=1)
    df['symptom_sum'] = df[['symptom_1','symptom_2','symptom_3','symptom_4','symptom_5']].sum(axis=1)

    df['defect_x_symptom']     = df['defect_sum'] * df['symptom_sum']
    df['any_defect']           = (df['defect_sum'] > 0).astype(int)
    df['any_symptom']          = (df['symptom_sum'] > 0).astype(int)
    df['high_symptom']         = (df['symptom_sum'] >= 4).astype(int)
    df['all_defects']          = (df['defect_sum'] == 4).astype(int)
    df['parent_age_gap']       = (df['father_age'] - df['mother_age']).abs()
    df['symptom_defect_ratio'] = df['symptom_sum'] / (df['defect_sum'] + 1)

    df['s4_and_s5']    = ((df['symptom_4'] == 1) & (df['symptom_5'] == 1)).astype(int)
    df['no_s4_s5']     = ((df['symptom_4'] == 0) & (df['symptom_5'] == 0)).astype(int)
    df['late_vs_early']= (df['symptom_4'].fillna(0) + df['symptom_5'].fillna(0)
                         - df['symptom_1'].fillna(0) - df['symptom_2'].fillna(0))
    df['weighted_sym'] = (df['symptom_1'].fillna(0)*1 + df['symptom_2'].fillna(0)*1 +
                          df['symptom_3'].fillna(0)*1 + df['symptom_4'].fillna(0)*2 +
                          df['symptom_5'].fillna(0)*2)

    df['both_parents_defect'] = ((df['mother_defect'] == 1) & (df['father_defect'] == 1)).astype(int)
    df['no_parent_defect']    = ((df['mother_defect'] == 0) & (df['father_defect'] == 0)).astype(int)

    return df


X_train = preprocess(TRAIN_DATA)
X_test  = preprocess(TEST_DATA)
y_train = TRAIN_LABEL['disorder'].values

print(f'X_train: {X_train.shape}, X_test: {X_test.shape}')

X_train: (13249, 59), X_test: (8834, 59)


In [5]:
# ── Cell 5: Class weights (a3 formula) ───────────────────────────────────────
class_counts  = np.bincount(y_train)
class_weights = len(y_train) / (10 * class_counts)

print('Class weights:')
for i, (n, w) in enumerate(zip(class_counts, class_weights)):
    print(f'  Class {i}: weight={w:.3f}  (n={n})')

Class weights:
  Class 0: weight=3.406  (n=389)
  Class 1: weight=0.641  (n=2068)
  Class 2: weight=1.216  (n=1090)
  Class 3: weight=0.428  (n=3096)
  Class 4: weight=22.843  (n=58)
  Class 5: weight=0.779  (n=1700)
  Class 6: weight=1.630  (n=813)
  Class 7: weight=0.501  (n=2643)
  Class 8: weight=14.559  (n=91)
  Class 9: weight=1.018  (n=1301)


In [6]:
# ── Cell 6: Train CatBoost — 3 seeds x 10 folds, save OOF probabilities ─────
# We MUST save OOF probabilities to calibrate thresholds on them
# Never calibrate on the same fold used for training — that would be cheating

all_oof_proba  = np.zeros((len(y_train), 10))
all_test_preds = np.zeros((len(X_test), 10))

for SEED in SEEDS:
    print(f"\n{'='*40} SEED={SEED} {'='*40}")
    skf = StratifiedKFold(n_splits=N_SPLITS, shuffle=True, random_state=SEED)

    oof_proba  = np.zeros((len(y_train), 10))
    test_preds = np.zeros((len(X_test), 10))
    fold_scores = []

    for fold, (tr_idx, val_idx) in enumerate(skf.split(X_train, y_train)):
        X_tr,  X_val = X_train.iloc[tr_idx], X_train.iloc[val_idx]
        y_tr,  y_val = y_train[tr_idx],      y_train[val_idx]

        model = CatBoostClassifier(
            iterations            = 2000,
            learning_rate         = 0.03,
            depth                 = 6,
            l2_leaf_reg           = 3,
            class_weights         = class_weights,
            early_stopping_rounds = 100,
            eval_metric           = 'Accuracy',
            random_seed           = SEED,
            verbose               = 0,
            thread_count          = -1,
        )
        model.fit(X_tr, y_tr, eval_set=(X_val, y_val), use_best_model=True)

        val_proba = model.predict_proba(X_val)
        score     = balanced_accuracy_score(y_val, np.argmax(val_proba, axis=1))
        fold_scores.append(score)
        print(f'  Fold {fold+1:2d}: BA={score:.4f}  best_iter={model.best_iteration_}')

        oof_proba[val_idx] += val_proba
        test_preds         += model.predict_proba(X_test) / N_SPLITS

    oof_score = balanced_accuracy_score(y_train, np.argmax(oof_proba, axis=1))
    print(f'  OOF BA (seed={SEED}): {oof_score:.4f} | mean={np.mean(fold_scores):.4f} ± {np.std(fold_scores):.4f}')

    all_oof_proba  += oof_proba  / len(SEEDS)
    all_test_preds += test_preds / len(SEEDS)

base_oof_score = balanced_accuracy_score(y_train, np.argmax(all_oof_proba, axis=1))
print(f'\nBASELINE OOF BA (no calibration): {base_oof_score:.4f}')


======================================== SEED=42 ========================================
  Fold  1: BA=0.3791  best_iter=146
  Fold  2: BA=0.4142  best_iter=31
  Fold  3: BA=0.3981  best_iter=26
  Fold  4: BA=0.3728  best_iter=282
  Fold  5: BA=0.3991  best_iter=2
  Fold  6: BA=0.4520  best_iter=137
  Fold  7: BA=0.3511  best_iter=30
  Fold  8: BA=0.3710  best_iter=45
  Fold  9: BA=0.4241  best_iter=55
  Fold 10: BA=0.4243  best_iter=51
  OOF BA (seed=42): 0.3984 | mean=0.3986 ± 0.0291

======================================== SEED=7 ========================================
  Fold  1: BA=0.3853  best_iter=175
  Fold  2: BA=0.4051  best_iter=117
  Fold  3: BA=0.4169  best_iter=168
  Fold  4: BA=0.4145  best_iter=123
  Fold  5: BA=0.3705  best_iter=18
  Fold  6: BA=0.4045  best_iter=81
  Fold  7: BA=0.3728  best_iter=42
  Fold  8: BA=0.3782  best_iter=27
  Fold  9: BA=0.4131  best_iter=186
  Fold 10: BA=0.3779  best_iter=78
  OOF BA (seed=7): 0.3942 | mean=0.3939 ± 0.0177

============

In [7]:
# ── Cell 7: Threshold calibration ────────────────────────────────────────────
# How it works:
#   Normally we predict: argmax(proba)
#   With thresholds:     argmax(proba * threshold_per_class)
#   Multiplying a class's proba by a higher threshold makes the model
#   more willing to predict that class — boosting its recall
#   We optimise the 10 threshold values on OOF probabilities

def predict_with_thresholds(proba, thresholds):
    """Multiply each class probability by its threshold, then take argmax."""
    return np.argmax(proba * thresholds, axis=1)


def neg_balanced_accuracy(thresholds, proba, y_true):
    """Objective function — minimise negative BA."""
    preds = predict_with_thresholds(proba, thresholds)
    return -balanced_accuracy_score(y_true, preds)


print('Baseline (argmax, no thresholds):')
base_preds = np.argmax(all_oof_proba, axis=1)
print(f'  OOF BA = {balanced_accuracy_score(y_train, base_preds):.4f}')

print('\nPer-class recall BEFORE calibration:')
disorder_names = {
    0:'레베르시', 1:'낭포성섬유증', 2:'당뇨', 3:'리증후군', 4:'암',
    5:'테이-삭스', 6:'혈색소침착증', 7:'사립체근병종', 8:'알츠하이머', 9:'확인안됨'
}
report_before = classification_report(y_train, base_preds, output_dict=True)
for cls in range(10):
    r = report_before[str(cls)]['recall']
    print(f'  Class {cls} ({disorder_names[cls]:12s}): {r:.3f}' + (' ← LOW' if r < 0.3 else ''))

# ── Optimise thresholds ──────────────────────────────────────────────────────
print('\nOptimising thresholds...')
best_result = None
best_ba     = base_oof_score

# Run multiple random starts to avoid local minima
np.random.seed(42)
n_starts = 50

for i in range(n_starts):
    # Random init: start near 1.0 with some perturbation
    init = np.random.uniform(0.5, 2.0, size=10)

    result = minimize(
        neg_balanced_accuracy,
        init,
        args=(all_oof_proba, y_train),
        method='Nelder-Mead',
        options={'maxiter': 2000, 'xatol': 1e-5, 'fatol': 1e-5}
    )

    if -result.fun > best_ba:
        best_ba     = -result.fun
        best_result = result
        print(f'  Start {i+1:3d}: NEW BEST OOF BA = {best_ba:.4f}')

if best_result is None:
    print('  No improvement found — thresholds all stay at 1.0')
    best_thresholds = np.ones(10)
else:
    best_thresholds = best_result.x
    # Normalise so max threshold = 1.0 (easier to interpret)
    best_thresholds = best_thresholds / best_thresholds.max()

print(f'\nBest thresholds found:')
for cls, t in enumerate(best_thresholds):
    direction = 'boost recall' if t > 1.0 else 'suppress' if t < 0.8 else 'neutral'
    print(f'  Class {cls} ({disorder_names[cls]:12s}): threshold={t:.4f}  ({direction})')

Baseline (argmax, no thresholds):
  OOF BA = 0.3860

Per-class recall BEFORE calibration:
  Class 0 (레베르시        ): 0.314
  Class 1 (낭포성섬유증      ): 0.392
  Class 2 (당뇨          ): 0.272 ← LOW
  Class 3 (리증후군        ): 0.351
  Class 4 (암           ): 0.759
  Class 5 (테이-삭스       ): 0.338
  Class 6 (혈색소침착증      ): 0.488
  Class 7 (사립체근병종      ): 0.237 ← LOW
  Class 8 (알츠하이머       ): 0.571
  Class 9 (확인안됨        ): 0.138 ← LOW

Optimising thresholds...
  Start  28: NEW BEST OOF BA = 0.3888

Best thresholds found:
  Class 0 (레베르시        ): threshold=0.9361  (neutral)
  Class 1 (낭포성섬유증      ): threshold=0.9776  (neutral)
  Class 2 (당뇨          ): threshold=0.9847  (neutral)
  Class 3 (리증후군        ): threshold=1.0000  (neutral)
  Class 4 (암           ): threshold=0.7648  (suppress)
  Class 5 (테이-삭스       ): threshold=0.7445  (suppress)
  Class 6 (혈색소침착증      ): threshold=0.7449  (suppress)
  Class 7 (사립체근병종      ): threshold=0.9010  (neutral)
  Class 8 (알츠하이머       ): threshold=0.8831  (neut

In [8]:
# ── Cell 8: Evaluate calibrated predictions on OOF ───────────────────────────
cal_oof_preds = predict_with_thresholds(all_oof_proba, best_thresholds)
cal_oof_ba    = balanced_accuracy_score(y_train, cal_oof_preds)

report_after = classification_report(y_train, cal_oof_preds, output_dict=True)

print(f'OOF BA before calibration: {base_oof_score:.4f}')
print(f'OOF BA after  calibration: {cal_oof_ba:.4f}  (Δ={cal_oof_ba - base_oof_score:+.4f})')

print('\nPer-class recall BEFORE vs AFTER calibration:')
print(f'{"Class":<5} {"Name":<14} {"Before":>8} {"After":>8} {"Change":>8}')
print('-' * 50)
for cls in range(10):
    r_before = report_before[str(cls)]['recall']
    r_after  = report_after[str(cls)]['recall']
    delta    = r_after - r_before
    flag     = ' ← improved' if delta > 0.02 else ' ← dropped' if delta < -0.02 else ''
    print(f'{cls:<5} {disorder_names[cls]:<14} {r_before:>8.3f} {r_after:>8.3f} {delta:>+8.3f}{flag}')

OOF BA before calibration: 0.3860
OOF BA after  calibration: 0.3888  (Δ=+0.0028)

Per-class recall BEFORE vs AFTER calibration:
Class Name             Before    After   Change
--------------------------------------------------
0     레베르시              0.314    0.337   +0.023 ← improved
1     낭포성섬유증            0.392    0.386   -0.006
2     당뇨                0.272    0.305   +0.032 ← improved
3     리증후군              0.351    0.440   +0.089 ← improved
4     암                 0.759    0.793   +0.034 ← improved
5     테이-삭스             0.338    0.188   -0.150 ← dropped
6     혈색소침착증            0.488    0.472   -0.016
7     사립체근병종            0.237    0.258   +0.021 ← improved
8     알츠하이머             0.571    0.549   -0.022 ← dropped
9     확인안됨              0.138    0.161   +0.023 ← improved


In [9]:
# ── Cell 9: Save both submissions — with and without calibration ─────────────
# Always save both so we can compare LB directly

# Without calibration (same as a3 logic)
raw_preds  = np.argmax(all_test_preds, axis=1)
sub_raw    = pd.DataFrame({'id': TEST_DATA.index, 'disorder': raw_preds}).set_index('id')
sub_raw.to_csv('submission_no_calibration.csv')
print('Saved: submission_no_calibration.csv')
print(sub_raw['disorder'].value_counts().sort_index())

print()

# With calibration
cal_preds  = predict_with_thresholds(all_test_preds, best_thresholds)
sub_cal    = pd.DataFrame({'id': TEST_DATA.index, 'disorder': cal_preds}).set_index('id')
sub_cal.to_csv('submission_calibrated.csv')
print('Saved: submission_calibrated.csv')
print(sub_cal['disorder'].value_counts().sort_index())

print(f'\nRecommend submitting calibrated version if OOF improvement > 0.005')
print(f'OOF improvement: {cal_oof_ba - base_oof_score:+.4f}')

Saved: submission_no_calibration.csv
disorder
0     382
1    1392
2     633
3    1562
4     248
5    1287
6    1081
7    1168
8     242
9     839
Name: count, dtype: int64

Saved: submission_calibrated.csv
disorder
0     386
1    1363
2     690
3    2102
4     261
5     605
6    1033
7    1192
8     198
9    1004
Name: count, dtype: int64

Recommend submitting calibrated version if OOF improvement > 0.005
OOF improvement: +0.0028
